In [1]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import from_json, col
from pyspark.sql.types import StructType, StructField, StringType, IntegerType, DoubleType
import time

print("Starting Spark Session for Kafka Streaming...")
spark = SparkSession.builder \
    .appName("EcommerceKafkaStreaming") \
    .getOrCreate()

spark.sparkContext.setLogLevel("WARN")

# Define the schema matching our producer
schema = StructType([
    StructField("event_time", StringType(), True),
    StructField("event_type", StringType(), True),
    StructField("product_id", IntegerType(), True),
    StructField("user_id", IntegerType(), True),
    StructField("price", DoubleType(), True)
])

# Read from Kafka. Remember, internal port is 29092
print("Connecting to Kafka...")
df = spark \
  .readStream \
  .format("kafka") \
  .option("kafka.bootstrap.servers", "kafka:29092") \
  .option("subscribe", "user_events") \
  .option("startingOffsets", "earliest") \
  .load()

Starting Spark Session for Kafka Streaming...
Connecting to Kafka...


In [2]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import from_json, col
from pyspark.sql.types import StructType, StructField, StringType, DoubleType, IntegerType

# 1. Start the SparkSession using the config from your docker-compose
spark = SparkSession.builder \
    .appName("PR1_Streaming_Demo") \
    .master("spark://spark-master:7077") \
    .getOrCreate()

# 2. Define the schema to parse the incoming JSON from Kafka
schema = StructType([
    StructField("event_time", StringType()),
    StructField("event_type", StringType()),
    StructField("product_id", IntegerType()),
    StructField("user_id", IntegerType()),
    StructField("price", DoubleType())
])

# 3. Read the stream from the internal Kafka listener
# NOTE: We use kafka:29092 because Spark is inside the Docker network 
df = spark.readStream \
    .format("kafka") \
    .option("kafka.bootstrap.servers", "kafka:29092") \
    .option("subscribe", "user_events") \
    .load()

# 4. Parse the JSON and select the columns
parsed_df = df.select(from_json(col("value").cast("string"), schema).alias("data")).select("data.*")

# 5. Write the stream to MinIO as Delta Lake 
query = parsed_df.writeStream \
    .format("delta") \
    .outputMode("append") \
    .option("checkpointLocation", "s3a://warehouse/checkpoints/user_events") \
    .start("s3a://warehouse/events")

print("✅ Streaming active! Check your MinIO console at localhost:9001")

✅ Streaming active! Check your MinIO console at localhost:9001


In [3]:
# Verification Script: Run this periodically during your demo
from pyspark.sql import SparkSession

# Connect to the existing session
spark = SparkSession.builder.getOrCreate()

# Read the Delta table from MinIO
# This satisfies Layer 2: Object Storage 
df_verification = spark.read.format("delta").load("s3a://warehouse/events")

print(f"📊 Total Records in Lakehouse: {df_verification.count()}")
print("Last 5 records ingested:")
df_verification.orderBy(col("event_time").desc()).show(5)

📊 Total Records in Lakehouse: 949
Last 5 records ingested:
+-------------------+----------+----------+-------+------+
|         event_time|event_type|product_id|user_id| price|
+-------------------+----------+----------+-------+------+
|2026-04-27 00:06:59|      cart|   6533072|    544| 117.3|
|2026-04-27 00:06:58|  purchase|   8830315|    701|299.55|
|2026-04-27 00:06:57|      view|   7242540|    909|450.98|
|2026-04-27 00:06:56|  purchase|   4389828|    148|399.29|
|2026-04-27 00:06:55|  purchase|   4424673|    816|136.02|
+-------------------+----------+----------+-------+------+
only showing top 5 rows

